# A — Data Preparation Pipeline:

In this notebook, we document the data preparation work for our phishing URL detection project.

We use this notebook to:
- inspect the raw dataset,
- confirm that it contains raw URL strings and binary labels,
- check missing values,
- check duplicate URLs,
- check conflicting labels,
- clean the dataset,
- create train/validation/test splits,
- prepare the data for feature engineering.

This step is important because the quality of the dataset directly affects the reliability of our models. In particular, duplicated URLs must be handled before splitting to avoid data leakage between the training and test sets.

## Import Libraries

In [82]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split

from urllib.parse import urlparse
from collections import Counter
import math
import re

from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

## Load Dataset

We load the dataset as it is from the raw data folder.

In [46]:
data_path = Path("../data/raw/mendeley_2026_commoncrawl_phishtank.csv")

In [47]:
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (149726, 2)
Columns: ['URL', 'Label']


,URL,Label
0,https://optus-myaccount.s2-tastewp.com/,1
1,https://www.googleapis.com/auth/drive.file';,0
2,https://wikipedia.org/,0
3,https://www.aol.com/2008-01-23-ask-the-dolans-...,0
4,https://rebrand.ly/uda4njv,1


## Quick Data Check

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149726 entries, 0 to 149725
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   URL     149726 non-null  str  
 1   Label   149726 non-null  int64
dtypes: int64(1), str(1)
memory usage: 12.6 MB


In [49]:
df.sample(5, random_state=42)

,URL,Label
125133,https://qcaprod.australiaeast.cloudapp.azure.c...,0
15585,https://usps.bbntnczlrw.top/,1
25447,http://www.theguardian.com/robots.txt,0
143062,https://pub-9ec1967945504971b7ee3cd05fc7a4f9.r...,1
48501,https://ipfs.eth.aragon.network/ipfs/bafkreidd...,1


The dataset has raw URL strings and numeric binary labels, which fits our project setup.

## Missing Values

In [50]:
df.isna().sum()

URL      0
Label    0
dtype: int64

No missing values were found, so no rows need to be removed for missing data.

## Label Distribution

In [51]:
df["Label"].value_counts().sort_index()

Label
0    94919
1    54807
Name: count, dtype: int64

In [52]:
df["Label"].value_counts(normalize=True).sort_index().round(4)

Label
0    0.634
1    0.366
Name: proportion, dtype: float64

#### Observation:
The dataset is imbalanced, but it still has both classes. Around 63.4% of the URLs are legitimate and 36.6% are phishing.

Later, we should use stratified splitting so the train, validation, and test sets keep similar class proportions.

## Duplicate Check

In [53]:
# We check duplicate URLs before splitting to avoid the same URL appearing in multiple sets.
df["URL"].duplicated().sum()

np.int64(19949)

There are 19,949 duplicate URL rows. We should remove duplicates before splitting so the same URL does not appear in both training and test data.

In [54]:
# Check if the same URL has more than one label.
df.groupby("URL")["Label"].nunique().gt(1).sum()

np.int64(0)

No conflicting labels were found. This means the same URL is not labeled as both legitimate and phishing.

## Cleaning Duplicate URLs

In [55]:
# Remove duplicate URLs before splitting.
df_clean = df.drop_duplicates(subset="URL").copy()

print("Original shape:", df.shape)
print("Clean shape:", df_clean.shape)
print("Removed rows:", len(df) - len(df_clean))

Original shape: (149726, 2)
Clean shape: (129777, 2)
Removed rows: 19949


After removing duplicate URLs, the dataset now has 129,777 unique URL records.

## Label Distribution After Cleaning

In [56]:
# Check class counts after removing duplicate URLs.
df_clean["Label"].value_counts().sort_index()

Label
0    74972
1    54805
Name: count, dtype: int64

In [57]:
# Check class ratios after removing duplicate URLs.
df_clean["Label"].value_counts(normalize=True).sort_index().round(4)

Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64

After duplicate removal, the cleaned dataset still contains both classes. The class ratio changed because many duplicate rows were from the legitimate class.

## Save Clean Dataset

In [58]:
# Save the cleaned dataset for the next notebooks.
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

clean_path = processed_dir / "clean_urls.csv"
df_clean.to_csv(clean_path, index=False)

print("Saved to:", clean_path)
print("Final shape:", df_clean.shape)

Saved to: ..\data\processed\clean_urls.csv
Final shape: (129777, 2)


The cleaned dataset was saved successfully and will be used in the next step for train/validation/test splitting.

## Dataset Audit Summary

In [59]:
# Final numbers for the data audit.
print("Raw rows:", len(df))
print("Clean rows:", len(df_clean))
print("Removed duplicate rows:", len(df) - len(df_clean))

print("\nClean label counts:")
print(df_clean["Label"].value_counts().sort_index())

print("\nClean label ratios:")
print(df_clean["Label"].value_counts(normalize=True).sort_index().round(4))

Raw rows: 149726
Clean rows: 129777
Removed duplicate rows: 19949

Clean label counts:
Label
0    74972
1    54805
Name: count, dtype: int64

Clean label ratios:
Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64


The raw dataset contained 149,726 rows. After checking duplicates and conflicting labels, we removed 19,949 duplicate URL rows and kept 129,777 unique URLs.

The cleaned dataset contains 74,972 legitimate URLs and 54,805 phishing URLs. This gives us a usable binary dataset for the next stage.

# Train / Validation / Test Split

## Split Configuration

In [60]:
RANDOM_STATE = 42

train_size = 0.70
val_size = 0.15
test_size = 0.15

## Create Stratified Splits

In [61]:
# First split: 70% train and 30% temporary data.
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    stratify=df_clean["Label"],
    random_state=RANDOM_STATE
)

In [62]:
# Second split: split the temporary data equally into validation and test.
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Label"],
    random_state=RANDOM_STATE
)

In [63]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (90843, 2)
Validation shape: (19467, 2)
Test shape: (19467, 2)


## Check Split Distributions

In [64]:
# Check label counts in each split.
print("Train:")
print(train_df["Label"].value_counts().sort_index())

print("\nValidation:")
print(val_df["Label"].value_counts().sort_index())

print("\nTest:")
print(test_df["Label"].value_counts().sort_index())

Train:
Label
0    52480
1    38363
Name: count, dtype: int64

Validation:
Label
0    11246
1     8221
Name: count, dtype: int64

Test:
Label
0    11246
1     8221
Name: count, dtype: int64


In [65]:
# Check label ratios in each split.
print("Train:")
print(train_df["Label"].value_counts(normalize=True).sort_index().round(4))

print("\nValidation:")
print(val_df["Label"].value_counts(normalize=True).sort_index().round(4))

print("\nTest:")
print(test_df["Label"].value_counts(normalize=True).sort_index().round(4))

Train:
Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64

Validation:
Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64

Test:
Label
0    0.5777
1    0.4223
Name: proportion, dtype: float64


The class ratios are the same across train, validation, and test sets. This confirms that stratified splitting worked as expected.

## Save Split Files

In [66]:
# Save the split datasets for modeling and feature engineering.
split_dir = Path("../data/splits")
split_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "val.csv", index=False)
test_df.to_csv(split_dir / "test.csv", index=False)

print("Saved train.csv:", train_df.shape)
print("Saved val.csv:", val_df.shape)
print("Saved test.csv:", test_df.shape)

Saved train.csv: (90843, 2)
Saved val.csv: (19467, 2)
Saved test.csv: (19467, 2)


The split files were saved successfully. We will use these files for feature engineering and modeling.

# Feature Engineering

## Helper Function: Shannon Entropy

- We will use Shannon entropy to measure how random or complex a URL string looks.

- In phishing detection, this can be useful because some phishing URLs contain random-looking characters, long tokens, or generated paths. Higher entropy does not always mean phishing, but it can be a useful signal when combined with other URL features.

In [67]:
def shannon_entropy(text):
    if not text:
        return 0
    
    counts = Counter(text)
    length = len(text)
    
    entropy = 0
    for count in counts.values():
        p = count / length
        entropy -= p * math.log2(p)
    
    return entropy

In [68]:
# Quick check with one URL from the dataset.
sample_url = train_df["URL"].iloc[0]

print(sample_url)
print("Entropy:", round(shannon_entropy(sample_url), 4))

https://chb-covidfaq-prod-7361.appspot.com/
Entropy: 4.2954


## URL Feature Extractor

We extract simple lexical and structural features from each URL.

These features describe the URL text itself, such as length, special characters, digits, HTTPS usage, IP address usage, and entropy.

In [69]:
suspicious_words = [
    "login", "secure", "account", "update", "verify",
    "bank", "paypal", "ebay", "signin", "password"
]

ip_pattern = r"^\d{1,3}(\.\d{1,3}){3}$"

In [70]:
def extract_url_features(url):
    url = str(url).strip()
    url_for_parse = url if "://" in url else "http://" + url
    parsed = urlparse(url_for_parse)
    
    hostname = parsed.netloc
    path = parsed.path
    query = parsed.query
    
    url_length = len(url)
    hostname_length = len(hostname)
    path_length = len(path)
    query_length = len(query)
    
    num_digits = sum(char.isdigit() for char in url)
    path_depth = len([part for part in path.split("/") if part])
    
    features = {
        "url_length": url_length,
        "hostname_length": hostname_length,
        "path_length": path_length,
        "query_length": query_length,
        "num_dots": url.count("."),
        "num_hyphens": url.count("-"),
        "num_underscores": url.count("_"),
        "num_slashes": url.count("/"),
        "num_at_symbol": url.count("@"),
        "num_question_marks": url.count("?"),
        "num_equals": url.count("="),
        "num_ampersand": url.count("&"),
        "num_percent": url.count("%"),
        "num_digits": num_digits,
        "digit_ratio": num_digits / url_length if url_length > 0 else 0,
        "has_https": int(url.lower().startswith("https://")),
        "has_ip_address": int(re.match(ip_pattern, hostname) is not None),
        "subdomain_depth": max(hostname.count(".") - 1, 0),
        "path_depth": path_depth,
        "hostname_has_hyphen": int("-" in hostname),
        "tld_in_path": int(any(tld in path.lower() for tld in [".com", ".net", ".org", ".edu", ".gov"])),
        "shannon_entropy": shannon_entropy(url),
        "suspicious_keyword_count": sum(word in url.lower() for word in suspicious_words)
    }
    
    return features

In [71]:
# Test the updated feature extractor on one URL.
sample_features = extract_url_features(sample_url)

sample_features

{'url_length': 43,
 'hostname_length': 34,
 'path_length': 1,
 'query_length': 0,
 'num_dots': 2,
 'num_hyphens': 3,
 'num_underscores': 0,
 'num_slashes': 3,
 'num_at_symbol': 0,
 'num_question_marks': 0,
 'num_equals': 0,
 'num_ampersand': 0,
 'num_percent': 0,
 'num_digits': 4,
 'digit_ratio': 0.09302325581395349,
 'has_https': 1,
 'has_ip_address': 0,
 'subdomain_depth': 1,
 'path_depth': 0,
 'hostname_has_hyphen': 1,
 'tld_in_path': 0,
 'shannon_entropy': 4.295353348118593,
 'suspicious_keyword_count': 0}

The updated feature extractor now returns 23 handcrafted URL features. These features are still simple and based only on the URL string, so they fit the scope of our data preparation work.

## Extract Features for Each Split

In [72]:
# Apply the feature extractor to each URL column.
train_features = train_df["URL"].apply(extract_url_features).apply(pd.Series)
val_features = val_df["URL"].apply(extract_url_features).apply(pd.Series)
test_features = test_df["URL"].apply(extract_url_features).apply(pd.Series)

In [73]:
print("Train features:", train_features.shape)
print("Validation features:", val_features.shape)
print("Test features:", test_features.shape)

Train features: (90843, 23)
Validation features: (19467, 23)
Test features: (19467, 23)


Each split now has 23 numeric URL features. The number of rows matches the original train, validation, and test splits, so the feature extraction step worked correctly.

## Add Labels to Feature Tables

In [74]:
# Add the target label back to each feature table.
train_features["Label"] = train_df["Label"].values
val_features["Label"] = val_df["Label"].values
test_features["Label"] = test_df["Label"].values

train_features.head()

,url_length,hostname_length,path_length,query_length,num_dots,num_hyphens,num_underscores,num_slashes,num_at_symbol,num_question_marks,...,digit_ratio,has_https,has_ip_address,subdomain_depth,path_depth,hostname_has_hyphen,tld_in_path,shannon_entropy,suspicious_keyword_count,Label
75917,43.0,34.0,1.0,0.0,2.0,3.0,0.0,3.0,0.0,0.0,...,0.093023,1.0,0.0,1.0,0.0,1.0,0.0,4.295353,0.0,1
8247,51.0,32.0,11.0,0.0,3.0,2.0,0.0,3.0,0.0,0.0,...,0.039216,1.0,0.0,1.0,1.0,1.0,0.0,4.177040,0.0,0
41474,76.0,14.0,54.0,0.0,2.0,3.0,0.0,7.0,0.0,0.0,...,0.052632,1.0,0.0,1.0,5.0,0.0,0.0,4.410537,0.0,0
22145,29.0,20.0,1.0,0.0,2.0,0.0,0.0,3.0,0.0,0.0,...,0.000000,1.0,0.0,1.0,0.0,0.0,0.0,3.650410,0.0,1
117791,28.0,19.0,1.0,0.0,2.0,0.0,0.0,3.0,0.0,0.0,...,0.000000,1.0,0.0,1.0,0.0,0.0,0.0,3.726474,0.0,1


In [75]:
print("Train:", train_features.shape)
print("Validation:", val_features.shape)
print("Test:", test_features.shape)

Train: (90843, 24)
Validation: (19467, 24)
Test: (19467, 24)


The feature tables now contain 23 handcrafted URL features plus the target label. The row counts still match the train, validation, and test splits.

In [77]:
# Check if feature extraction created any missing values.
print("Train missing values:", train_features.isna().sum().sum())
print("Validation missing values:", val_features.isna().sum().sum())
print("Test missing values:", test_features.isna().sum().sum())

Train missing values: 0
Validation missing values: 0
Test missing values: 0


No missing values were created during feature extraction, so the feature tables are ready to save.

## Save Feature Files

In [79]:
# Save unscaled feature tables for modeling.
feature_dir = Path("../data/features")
feature_dir.mkdir(parents=True, exist_ok=True)

train_features.to_csv(feature_dir / "train_features.csv", index=False)
val_features.to_csv(feature_dir / "val_features.csv", index=False)
test_features.to_csv(feature_dir / "test_features.csv", index=False)

In [80]:
print("Saved train_features.csv:", train_features.shape)
print("Saved val_features.csv:", val_features.shape)
print("Saved test_features.csv:", test_features.shape)

Saved train_features.csv: (90843, 24)
Saved val_features.csv: (19467, 24)
Saved test_features.csv: (19467, 24)


The unscaled feature files were saved successfully. These files are useful for models such as Random Forest, and they also let us keep a clear copy of the original extracted features.

# Feature Scaling

In [83]:
# Keep all columns except the target label.
feature_cols = [col for col in train_features.columns if col != "Label"]

print("Number of feature columns:", len(feature_cols))

Number of feature columns: 23


In [84]:
# Initialize Scaler
scaler = StandardScaler()

StandardScaler was used as the initial baseline because it is simple and commonly used for numerical ML models. Since URL features may contain outliers, RobustScaler could be tested later as an additional preprocessing variant, but we kept StandardScaler for the first reproducible pipeline.

In [85]:
# Fit Scaler on Training Data
train_scaled = train_features.copy()

# Fit only on training data to avoid data leakage.
train_scaled[feature_cols] = scaler.fit_transform(train_features[feature_cols])

train_scaled.head()

,url_length,hostname_length,path_length,query_length,num_dots,num_hyphens,num_underscores,num_slashes,num_at_symbol,num_question_marks,...,digit_ratio,has_https,has_ip_address,subdomain_depth,path_depth,hostname_has_hyphen,tld_in_path,shannon_entropy,suspicious_keyword_count,Label
75917,-0.278316,0.963660,-0.702299,-0.238061,-0.315821,0.439718,-0.23579,-0.692160,-0.105251,-0.500442,...,0.034901,0.364113,-0.057754,0.052092,-1.028359,1.822282,-0.099356,-0.260562,-0.210616,1
8247,-0.215207,0.816247,-0.448899,-0.238061,0.364427,0.113216,-0.23579,-0.692160,-0.105251,-0.500442,...,-0.497517,0.364113,-0.057754,0.052092,-0.490397,1.822282,-0.099356,-0.515406,-0.210616,0
41474,-0.017989,-0.510473,0.640719,-0.238061,-0.315821,0.439718,-0.23579,1.360912,-0.105251,-0.500442,...,-0.364769,0.364113,-0.057754,0.052092,1.661450,-0.548762,-0.099356,-0.012458,-0.210616,0
22145,-0.388759,-0.068233,-0.702299,-0.238061,-0.315821,-0.539790,-0.23579,-0.692160,-0.105251,-0.500442,...,-0.885550,0.364113,-0.057754,0.052092,-1.028359,-0.548762,-0.099356,-1.649753,-0.210616,1
117791,-0.396647,-0.141940,-0.702299,-0.238061,-0.315821,-0.539790,-0.23579,-0.692160,-0.105251,-0.500442,...,-0.885550,0.364113,-0.057754,0.052092,-1.028359,-0.548762,-0.099356,-1.485914,-0.210616,1


In [86]:
# Transform Validation and Test Data
val_scaled = val_features.copy()
test_scaled = test_features.copy()

val_scaled[feature_cols] = scaler.transform(val_features[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_features[feature_cols])

print("Train scaled:", train_scaled.shape)
print("Validation scaled:", val_scaled.shape)
print("Test scaled:", test_scaled.shape)

Train scaled: (90843, 24)
Validation scaled: (19467, 24)
Test scaled: (19467, 24)


The scaled feature tables keep the same shape as the unscaled feature tables. The scaler was fitted only on the training data, then applied to validation and test data to avoid data leakage.

## Save Scaled Feature Files

In [ ]:
# Save scaled feature tables for models that need normalized features.
scaled_dir = Path("../data/features_scaled")
scaled_dir.mkdir(parents=True, exist_ok=True)

train_scaled.to_csv(scaled_dir / "train_features_scaled.csv", index=False)
val_scaled.to_csv(scaled_dir / "val_features_scaled.csv", index=False)
test_scaled.to_csv(scaled_dir / "test_features_scaled.csv", index=False)

In [88]:
print("Saved train_features_scaled.csv:", train_scaled.shape)
print("Saved val_features_scaled.csv:", val_scaled.shape)
print("Saved test_features_scaled.csv:", test_scaled.shape)

Saved train_features_scaled.csv: (90843, 24)
Saved val_features_scaled.csv: (19467, 24)
Saved test_features_scaled.csv: (19467, 24)


# Final Summary

In [89]:
print("Raw dataset:", df.shape)
print("Clean dataset:", df_clean.shape)

print("\nTrain split:", train_df.shape)
print("Validation split:", val_df.shape)
print("Test split:", test_df.shape)

print("\nUnscaled feature files:")
print("Train:", train_features.shape)
print("Validation:", val_features.shape)
print("Test:", test_features.shape)

print("\nScaled feature files:")
print("Train:", train_scaled.shape)
print("Validation:", val_scaled.shape)
print("Test:", test_scaled.shape)

Raw dataset: (149726, 2)
Clean dataset: (129777, 2)

Train split: (90843, 2)
Validation split: (19467, 2)
Test split: (19467, 2)

Unscaled feature files:
Train: (90843, 24)
Validation: (19467, 24)
Test: (19467, 24)

Scaled feature files:
Train: (90843, 24)
Validation: (19467, 24)
Test: (19467, 24)


## Handcrafted Feature Set

We extracted 23 handcrafted URL features from the raw URL string.

| Feature group | Features |
|---|---|
| Length-based | `url_length`, `hostname_length`, `path_length`, `query_length` |
| Character counts | `num_dots`, `num_hyphens`, `num_underscores`, `num_slashes`, `num_at_symbol`, `num_question_marks`, `num_equals`, `num_ampersand`, `num_percent`, `num_digits` |
| Ratios / complexity | `digit_ratio`, `shannon_entropy` |
| URL structure | `has_https`, `has_ip_address`, `subdomain_depth`, `path_depth`, `hostname_has_hyphen`, `tld_in_path` |
| Keyword signal | `suspicious_keyword_count` |

In [91]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "legitimate_count": [
        (train_df["Label"] == 0).sum(),
        (val_df["Label"] == 0).sum(),
        (test_df["Label"] == 0).sum()
    ],
    "phishing_count": [
        (train_df["Label"] == 1).sum(),
        (val_df["Label"] == 1).sum(),
        (test_df["Label"] == 1).sum()
    ],
    "phishing_ratio": [
        (train_df["Label"] == 1).mean(),
        (val_df["Label"] == 1).mean(),
        (test_df["Label"] == 1).mean()
    ]
})

split_summary

,split,rows,legitimate_count,phishing_count,phishing_ratio
0,train,90843,52480,38363,0.422300
1,validation,19467,11246,8221,0.422304
2,test,19467,11246,8221,0.422304


In [92]:
report_dir = Path("../reports/tables")
report_dir.mkdir(parents=True, exist_ok=True)

split_summary.to_csv(report_dir / "split_summary.csv", index=False)